# AG952 | Workshop 9 -- BrewDog: Reading the Signals

**Textual Analytics for Accounting and Finance**
University of Strathclyde Business School

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG952/blob/main/materials/week09/notebooks/AG952_W9_BrewDog_Transformer_Student.ipynb)

---

## Your role

You are a financial news analyst at a portfolio management firm. The investment committee holds a position in BrewDog plc and has asked whether UK newspaper coverage over the period 2010 to 2025 contained early warning signals of the company's difficulties -- information that, applied systematically, might have informed decisions about the position at key points in the timeline.

You have access to a corpus of **1,011 articles** drawn from UK national newspapers via Factiva. These include reporting from The Sun, The Times, the Financial Times and the Daily Mail, covering BrewDog from its founding through to the period immediately after its entry into administration. Your task is to apply systematic text analysis methods to investigate the following questions.

---

## Research questions

**RQ1.** How did the volume and topic composition of BrewDog coverage change across its growth, controversy and crisis phases, and do the topic models recover the narrative shifts that are known from other sources?

**RQ2.** Do dictionary-based and transformer-based sentiment measures agree on the direction and timing of sentiment shifts, and what explains any divergence between methods?

**RQ3.** Can word-level explanations from transformer models -- produced through integrated gradients -- provide signals reliable enough to inform financial analysis?

---

| Section | Checkpoint | Research question | Estimated time |
|---|---|---|---|
| Setup | CP0 -- Team registration | -- | 2 min |
| The corpus | CP1 -- Descriptive statistics | RQ1 | 5 min |
| Topic modelling | CP2 -- Pre-processing + CP3 -- LDA by era | RQ1 | 18 min |
| Sentiment analysis | CP4 -- Dictionary and transformer sentiment | RQ2 | 10 min |
| Explainability | CP5 -- Integrated gradients | RQ3 | 7 min |
| Submission | CP6 -- Analyst's note | All | 3 min |

> **Before Part 4:** CP5 downloads a pre-trained model (approximately 420 MB on first run). Start the download cell as soon as you reach the explainability section -- inference on CPU takes two to three minutes and you can read the context notes while it runs.


In [ ]:
#@title Configuration -- instructor sets these before the session
APPS_SCRIPT_URL = ""  # paste deployed Apps Script URL here
DATA_URL     = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week09/data/brewdog_articles_factiva.csv"
LM_NEG_URL   = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week07/data/lm_negative.txt"
LM_POS_URL   = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week07/data/lm_positive.txt"
HIV4_NEG_URL = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week07/data/harvard_iv4_negative.txt"
HIV4_POS_URL = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week07/data/harvard_iv4_positive.txt"


In [ ]:
#@title Step 0 -- Install and import dependencies (run first, approximately 90 seconds)

import sys, subprocess

# ── Phase 1: numpy version check ─────────────────────────────────────────
# captum (used by transformers-interpret) is compiled against numpy 1.x.
# On a fresh Colab runtime, numpy 2.x is the default.
#
# If numpy 2.x is detected:
#   1. numpy<2.0 is installed into the environment.
#   2. The kernel is killed so Colab restarts it (this is the only reliable
#      way to reload numpy in Colab -- raise SystemExit is caught by IPython).
#   3. After the session reconnects, re-run the Configuration cell above
#      this one, then re-run this Step 0 cell. The restart block will be
#      skipped and installation will complete normally.

try:
    import numpy as _np_pre
    _np_major = int(_np_pre.__version__.split(".")[0])
except ImportError:
    _np_major = 2

if _np_major >= 2:
    print(f"numpy {_np_pre.__version__} detected -- downgrading to 1.x (captum compatibility)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy<2.0", "-q"], check=True)
    print("Downgrade installed.")
    print()
    print("Restarting runtime now.")
    print("After the session reconnects:")
    print("  1. Re-run the Configuration cell (the cell above this one)")
    print("  2. Re-run this Step 0 cell")
    import os, signal
    os.kill(os.getpid(), signal.SIGKILL)

# ── Phase 2: install remaining packages ──────────────────────────────────
# Reached only when numpy 1.x is already active (second run, or GPU runtime).
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "transformers-interpret", "captum", "transformers", "torch",
     "wordcloud", "vaderSentiment", "scikit-learn", "nltk",
     "requests", "pandas", "matplotlib", "seaborn", "ipywidgets", "-q"],
    check=True,
)

import numpy as _np_v
print(f"numpy {_np_v.__version__} (1.x confirmed -- ready to proceed)")

# ── Standard imports ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import re, json, time, warnings, requests, random, collections
from IPython.display import display, clear_output, Markdown, HTML

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

import nltk
for pkg in ["stopwords", "punkt", "wordnet", "averaged_perceptron_tagger"]:
    nltk.download(pkg, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import ipywidgets as widgets
from wordcloud import WordCloud
warnings.filterwarnings("ignore")

# ── Session data dict ─────────────────────────────────────────────────────
GSHEET_COLUMNS = [
    "team_name", "cp1_period",
    "cp2_normalisation", "cp2_stopwords", "cp2_remove_numbers", "cp2_lowercase",
    "cp3_n_topics", "cp3_topic_labels",
    "cp4_dictionary", "cp4_sentiment_2010_2014", "cp4_sentiment_2019_2025",
    "cp6_model", "cp6_finbert_accuracy", "cp6_distilbert_accuracy",
    "cp7_interpretability_choice", "cp8_analyst_note", "timestamp",
]
session_data = {col: None for col in GSHEET_COLUMNS}

def validate_checkpoint(required_globals, checkpoint_name):
    missing = [v for v in required_globals if globals().get(v) is None]
    if missing:
        raise RuntimeError(
            f"Complete {checkpoint_name} before running this cell. "
            f"Still needed: {', '.join(missing)}"
        )

def load_wordlist(url):
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        words = set(r.text.lower().split())
        return words
    except Exception as e:
        print(f"  Warning: could not load wordlist from {url} ({e})")
        return set()

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

print("Dependencies loaded. Run CP0 to register your team.")

## Checkpoint 0 — Team Registration


In [ ]:
#@title CP0 — Register your team
# Instructor pre-configures TEAM_ASSIGNMENT before the session.
# All teams analyse the same BrewDog corpus but make independent methodological choices.

TEAM_ASSIGNMENT = {
    "The Disclosure Desk":   "BrewDog",
    "Alpha Analysts":        "BrewDog",
    "The Write-Offs":        "BrewDog",
    "Narrative Risk":        "BrewDog",
    "Sentiment Squad":       "BrewDog",
    "The Fog Index":         "BrewDog",
    "Punk Analysts":         "BrewDog",
    "Bear Market Brewers":   "BrewDog",
    "Item 1A-OK":            "BrewDog",
    "The Quant Collective":  "BrewDog",
    "FinText Co.":           "BrewDog",
    "Signals & Noise":       "BrewDog",
}

_cp0_team_w = widgets.Dropdown(
    options=["-- select your team --"] + sorted(TEAM_ASSIGNMENT.keys()),
    description="Team:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="420px"),
)
_cp0_out = widgets.Output()
_cp0_btn = widgets.Button(
    description="Register team",
    button_style="primary",
    layout=widgets.Layout(width="180px"),
)

def _run_cp0(b):
    global _cp0_done
    _cp0_out.clear_output(wait=True)
    with _cp0_out:
        if _cp0_team_w.value == "-- select your team --":
            print("Select your team name first.")
            return
        session_data["team_name"] = _cp0_team_w.value
        _cp0_done = True
        print(f"Registered: {_cp0_team_w.value}")
        print(f"Subject: BrewDog plc — news corpus analysis, 2010–2025")
        print(f"\nObjective: Examine how financial and general news coverage of BrewDog")
        print(f"evolved from its founding as a craft-beer disruptor through to its")
        print(f"entry into administration in 2025.")
        print(f"\nCP0 complete. Run CP1 to load the corpus.")

_cp0_btn.on_click(_run_cp0)
display(widgets.VBox([_cp0_team_w, _cp0_btn, _cp0_out]))


In [ ]:
#@title CP1 -- Load corpus and explore coverage patterns

validate_checkpoint(["_cp0_done"], "CP0")

# Use DATA_URL if the Config cell was run; fall back to the hardcoded URL otherwise.
# (After a Step 0 restart the Config cell must be re-run, but this ensures CP1
# does not crash with a NameError if the student forgot.)
_CORPUS_URL = globals().get(
    "DATA_URL",
    "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week09/data/brewdog_articles_factiva.csv",
)

try:
    articles = pd.read_csv(_CORPUS_URL)
    print(f"Corpus loaded: {len(articles)} articles, "
          f"{articles['year'].nunique()} years "
          f"({int(articles['year'].min())} to {int(articles['year'].max())})")
except Exception as e:
    print(f"Could not load corpus ({e})")
    print(f"URL attempted: {_CORPUS_URL}")
    print("Check that the Configuration cell has been run and that your runtime has internet access.")
    raise

articles["date"] = pd.to_datetime(articles["date"], errors="coerce")
articles["year"] = articles["year"].astype(int)

_cp1_out = widgets.Output()
_cp1_btn = widgets.Button(
    description="Load corpus",
    button_style="primary",
    layout=widgets.Layout(width="180px"),
)

def _run_cp1(b):
    global _cp1_done, working_articles, _period_label
    _cp1_out.clear_output(wait=True)
    with _cp1_out:
        # Always use the full timeline — era-level analysis is handled in CP3 and CP4
        working_articles = articles.copy()
        _period_label = "Full timeline (2010-2025)"
        session_data["cp1_period"] = _period_label
        _cp1_done = True

        n = len(working_articles)
        yr_min, yr_max = int(working_articles["year"].min()), int(working_articles["year"].max())
        print(f"Working corpus: {n} articles, {yr_min}\u2013{yr_max} (full timeline)")
        print(f"Mean article length: {working_articles['word_count'].mean():.0f} words  "
              f"(min {working_articles['word_count'].min()}, max {working_articles['word_count'].max()})")

        # --- Plot 1: articles by year, colour-coded by narrative era ---
        ERA_COLOURS = {
            "Growth":         "#22c55e",
            "Scaling":        "#f59e0b",
            "Controversy":    "#f97316",
            "Decline":        "#ef4444",
            "Administration": "#dc2626",
        }
        def era_colour(y):
            if y <= 2014: return ERA_COLOURS["Growth"]
            if y <= 2018: return ERA_COLOURS["Scaling"]
            if y <= 2021: return ERA_COLOURS["Controversy"]
            if y <= 2023: return ERA_COLOURS["Decline"]
            return ERA_COLOURS["Administration"]

        yr_counts = working_articles.groupby("year").size()
        bar_colors = [era_colour(y) for y in yr_counts.index]

        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        fig.suptitle("BrewDog corpus -- coverage overview", fontsize=13, fontweight="bold", y=1.01)

        # Top-left: articles by year
        axes[0, 0].bar(yr_counts.index, yr_counts.values, color=bar_colors, edgecolor="white", linewidth=0.5)
        from matplotlib.patches import Patch
        era_legend = [Patch(facecolor=c, label=e) for e, c in ERA_COLOURS.items()]
        axes[0, 0].set_title("Articles per year")
        axes[0, 0].set_xlabel("Year")
        axes[0, 0].set_ylabel("Article count")
        axes[0, 0].tick_params(axis="x", rotation=45)
        axes[0, 0].legend(handles=era_legend, title="Era", fontsize=7, loc="upper left")

        # Key event annotations on articles-per-year chart
        _CP1_EVENTS = [
            (2017, "Jun 2017\n\u00a3213m TSG\ninvestment",  "#15803d"),
            (2021, "Jun 2021\nCulture open\nletter",        "#dc2626"),
            (2022, "Dec 2022\nGoing-concern\naudit",        "#7c3aed"),
            (2024, "Mar 2024\nWatt departs\nas CEO",        "#1d4ed8"),
        ]
        _y_top = yr_counts.values.max()
        for _ev_year, _ev_label, _ev_color in _CP1_EVENTS:
            axes[0, 0].axvline(_ev_year, color=_ev_color, linewidth=1.4,
                               linestyle=":", alpha=0.85, zorder=5)
            axes[0, 0].text(_ev_year + 0.15, _y_top * 0.97, _ev_label,
                            ha="left", va="top", fontsize=6,
                            color=_ev_color, linespacing=1.3)

        # Top-right: articles by newspaper (top 8)
        top_papers = working_articles["newspaper"].value_counts().head(8)
        paper_colors = plt.cm.tab10(np.linspace(0, 1, len(top_papers)))
        axes[0, 1].barh(
            top_papers.index[::-1], top_papers.values[::-1],
            color=paper_colors[::-1], edgecolor="white", linewidth=0.5,
        )
        axes[0, 1].set_title("Articles by newspaper (top 8)")
        axes[0, 1].set_xlabel("Article count")
        axes[0, 1].tick_params(axis="y", labelsize=8)

        # Bottom-left: article word count distribution
        axes[1, 0].hist(
            working_articles["word_count"].clip(upper=2000), bins=30,
            color="#4f46e5", edgecolor="white", alpha=0.85,
        )
        axes[1, 0].set_title("Article length distribution (words, capped at 2,000)")
        axes[1, 0].set_xlabel("Word count")
        axes[1, 0].set_ylabel("Articles")
        axes[1, 0].axvline(
            working_articles["word_count"].median(), color="#ef4444",
            linestyle="--", linewidth=1.2, label=f"Median: {working_articles['word_count'].median():.0f}",
        )
        axes[1, 0].legend(fontsize=8)

        # Bottom-right: mean article length by era
        working_articles["era"] = working_articles["year"].apply(
            lambda y: "Growth\n2010-14" if y <= 2014 else
                      ("Scaling\n2015-18" if y <= 2018 else
                       ("Controversy\n2019-21" if y <= 2021 else
                        ("Decline\n2022-23" if y <= 2023 else "Admin\n2024-25")))
        )
        era_wc = working_articles.groupby("era", sort=False)["word_count"].mean()
        era_order = ["Growth\n2010-14", "Scaling\n2015-18", "Controversy\n2019-21",
                     "Decline\n2022-23", "Admin\n2024-25"]
        era_wc = era_wc.reindex([e for e in era_order if e in era_wc.index])
        bar_colours_era = [ERA_COLOURS["Growth"], ERA_COLOURS["Scaling"],
                           ERA_COLOURS["Controversy"], ERA_COLOURS["Decline"],
                           ERA_COLOURS["Administration"]][:len(era_wc)]
        axes[1, 1].bar(
            range(len(era_wc)), era_wc.values,
            color=bar_colours_era, edgecolor="white", linewidth=0.5,
        )
        axes[1, 1].set_xticks(range(len(era_wc)))
        axes[1, 1].set_xticklabels(era_wc.index, fontsize=8)
        axes[1, 1].set_title("Mean article length by narrative era")
        axes[1, 1].set_ylabel("Mean word count")
        for i, v in enumerate(era_wc.values):
            axes[1, 1].text(i, v + 5, f"{v:.0f}", ha="center", fontsize=9)

        plt.tight_layout()
        plt.show()

        # Summary stats table
        print("\n  Coverage summary by narrative era:")
        print(f"  {'Era':<26} {'Articles':>8}  {'% of total':>10}  {'Mean words':>10}")
        print(f"  {'-'*58}")
        for era in era_order:
            mask = working_articles["era"] == era
            cnt = mask.sum()
            pct = cnt / n * 100
            mwc = working_articles.loc[mask, "word_count"].mean()
            if cnt > 0:
                print(f"  {era.replace(chr(10), ' '):<26} {cnt:>8,}  {pct:>9.1f}%  {mwc:>10.0f}")

        print(f"\nCP1 complete. Note the coverage spike in 2021 (culture controversy) and the")
        print(f"volume of crisis-era articles -- this will shape your sentiment measurements.")
        print("Run the CP1 Feedback cell, then proceed to CP2.")

_cp1_btn.on_click(_run_cp1)
display(widgets.VBox([_cp1_btn, _cp1_out]))


In [ ]:
#@title CP1 — Feedback (run after clicking Load corpus)
validate_checkpoint(["_cp1_done"], "CP1")
display(Markdown(
    "### CP1 Feedback — The BrewDog Story: From Punk Pioneer to Administration\n\n"
    "BrewDog was founded in Fraserburgh, Scotland in 2007 by James Watt and Martin Dickie. "
    "Its **Equity for Punks** crowdfunding model — which ultimately raised over £75m from more than 200,000 small "
    "investors — was widely covered as a disruptive innovation in both the craft beer and corporate finance press.\n\n"
    "**The corpus you have just loaded** contains 1,011 articles drawn from UK national newspapers — predominantly "
    "The Sun, The Times, the Financial Times, the Daily Mail, and The Guardian — covering BrewDog from 2010 to early "
    "2026. Note the source mix: tabloids (The Sun, Daily Mail) cover BrewDog primarily through a consumer and "
    "controversy lens; financial press (FT, FT.com) focus on investment, accounts, and financial distress. "
    "This source diversity is not neutral — it shapes what the corpus 'knows' about BrewDog.\n\n"
    "**Five distinct phases** are visible in the volume chart:\n\n"
    "**2010–2014 — The growth narrative.** Coverage focused on product innovation, anti-establishment brand "
    "positioning, and the mechanics of equity crowdfunding as a novel financing instrument. Article counts "
    "were low but tone was broadly celebratory.\n\n"
    "**2015–2018 — Complexity and scale.** A £213m minority investment by TSG Consumer Partners in 2017 attracted "
    "scrutiny about whether BrewDog had abandoned its punk credentials. Coverage broadened from beer trade to "
    "national business press.\n\n"
    "**2019–2021 — Reputational controversy.** An open letter signed by 61 former employees in June 2021 "
    "alleging a 'culture of fear' triggered the largest sustained coverage spike in the corpus.\n\n"
    "**2022–2023 — Financial decline.** Declining revenues, going-concern audit qualifications and bar closures "
    "dominate coverage.\n\n"
    "**2024–2025 — Administration.** Entry into administration completed the arc — and explains the "
    "sustained dark red bars in your chart.\n\n"
    "> **Your analytical task:** Use the methods in this workshop to trace how this narrative is captured — and "
    "sometimes missed — by computational text analysis."
))
print("Proceed to CP2.")


## Part 2 -- Topic Modelling


In [ ]:
#@title CP2 — Pre-processing choices

validate_checkpoint(["_cp1_done"], "CP1")

_cp2_norm_w = widgets.Dropdown(
    options=["None", "Porter Stemmer", "Lemmatisation (WordNetLemmatizer)"],
    value="Lemmatisation (WordNetLemmatizer)",
    description="Normalisation:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="460px"),
)
_cp2_sw_w = widgets.Dropdown(
    options=[
        "NLTK standard",
        "Finance-adjusted (retain modals & negation)",
        "None",
    ],
    value="Finance-adjusted (retain modals & negation)",
    description="Stop words:",
    style={"description_width": "180px"},
    layout=widgets.Layout(width="460px"),
)
_cp2_num_w = widgets.RadioButtons(
    options=["Yes", "No"],
    value="Yes",
    description="Remove numbers:",
    style={"description_width": "180px"},
)
_cp2_lower_w = widgets.RadioButtons(
    options=["Yes", "No"],
    value="Yes",
    description="Lowercase:",
    style={"description_width": "180px"},
)
_cp2_out = widgets.Output()
_cp2_btn = widgets.Button(
    description="Apply pipeline",
    button_style="primary",
    layout=widgets.Layout(width="180px"),
)

MODALS = {"will", "may", "might", "could", "should", "would", "must", "shall", "can"}
NEGATION = {"not", "no", "nor", "never", "neither", "without", "cannot"}


def _apply_pipeline(text):
    t = str(text)
    if _cp2_lower_w.value == "Yes":
        t = t.lower()
    if _cp2_norm_w.value == "Lemmatisation (WordNetLemmatizer)":
        lem = WordNetLemmatizer()
        tokens = [lem.lemmatize(w) for w in re.findall(r"[a-zA-Z]+", t)]
    elif _cp2_norm_w.value == "Porter Stemmer":
        stem = PorterStemmer()
        tokens = [stem.stem(w) for w in re.findall(r"[a-zA-Z]+", t)]
    else:
        tokens = re.findall(r"[a-zA-Z]+", t)
    if _cp2_num_w.value == "Yes":
        tokens = [w for w in tokens if w.isalpha()]
    sw = set(stopwords.words("english"))
    if _cp2_sw_w.value == "Finance-adjusted (retain modals & negation)":
        sw = sw - MODALS - NEGATION
    elif _cp2_sw_w.value == "None":
        sw = set()
    tokens = [w for w in tokens if w.lower() not in sw]
    # Remove 1-2 character tokens (catches lemmatisation artefacts: "wa"<-was, "ha"<-has)
    tokens = [w for w in tokens if len(w) >= 3]
    return tokens


def _run_cp2(b):
    global _cp2_done, processed_docs, vocab_size
    _cp2_out.clear_output(wait=True)
    with _cp2_out:
        session_data.update(
            {
                "cp2_normalisation": _cp2_norm_w.value,
                "cp2_stopwords": _cp2_sw_w.value,
                "cp2_remove_numbers": _cp2_num_w.value,
                "cp2_lowercase": _cp2_lower_w.value,
            }
        )
        working_articles["tokens"] = working_articles["text"].apply(_apply_pipeline)
        working_articles["token_str"] = working_articles["tokens"].apply(" ".join)
        processed_docs = working_articles["token_str"].tolist()
        all_tokens = [t for toks in working_articles["tokens"] for t in toks]
        vocab_size = len(set(all_tokens))

        # Word cloud
        freq = collections.Counter(all_tokens)
        wc = WordCloud(
            width=800, height=300, background_color="white",
            colormap="Blues", max_words=100,
        ).generate_from_frequencies(freq)
        fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
        axes[0].imshow(wc, interpolation="bilinear")
        axes[0].axis("off")
        axes[0].set_title("Top terms across corpus")

        # Tokens per doc distribution
        doc_lengths = working_articles["tokens"].apply(len)
        axes[1].hist(doc_lengths, bins=25, color="#4f46e5", edgecolor="white", alpha=0.85)
        axes[1].set_title("Token count per article")
        axes[1].set_xlabel("Tokens")
        axes[1].set_ylabel("Articles")
        plt.tight_layout()
        plt.show()

        print(f"Pipeline applied. Vocabulary size: {vocab_size:,} unique tokens")
        print(
            f"Mean tokens per article: {doc_lengths.mean():.0f} "
            f"(min {doc_lengths.min()}, max {doc_lengths.max()})"
        )
        sample = working_articles["tokens"].iloc[0][:20]
        print(f"\nSample (first 20 tokens of article 0): {sample}")

        # --- Vocabulary consequence: show the impact of the stop-word choice ---
        print("\n" + "─" * 60)
        print("Vocabulary consequence: stop-word choice and negation/modal words")
        print("─" * 60)
        _sw_standard = set(stopwords.words("english"))
        _neg_in_std   = sorted(w for w in NEGATION if w in _sw_standard)
        _modal_in_std = sorted(w for w in MODALS   if w in _sw_standard)
        print(f"\nNLTK standard stop list removes these negation words: {_neg_in_std}")
        print(f"NLTK standard also removes these modal verbs:         {_modal_in_std}")

        # Count occurrences in raw corpus using combined text (fast approximate count)
        _combined = " " + " ".join(working_articles["text"].fillna("").str.lower().tolist()) + " "
        _target_words = sorted(NEGATION | MODALS)
        _raw_counts = {w: _combined.count(f" {w} ") for w in _target_words}
        _top_counts = sorted(
            ((w, c) for w, c in _raw_counts.items() if c > 0),
            key=lambda x: -x[1],
        )[:10]

        print(f"\nCorpus frequency (raw text, top 10 negation/modal words):")
        for _word, _cnt in _top_counts:
            _status = "REMOVED by NLTK std" if _word in _sw_standard else "kept by all choices"
            print(f"  {_word:<12} {_cnt:>6,} occurrences   [{_status}]")

        _current_sw = _cp2_sw_w.value
        print(f"\nYour choice: '{_current_sw}'")
        if "Finance-adjusted" in _current_sw:
            print("  Negation and modal words are RETAINED in your processed documents.")
            print("  'not', 'never', 'without', 'cannot' appear in your LDA vocabulary.")
        elif "NLTK standard" in _current_sw:
            print("  Negation and modal words have been REMOVED from your processed documents.")
            print("  The sentence 'The company did not benefit from the restructuring'")
            print("  becomes 'company benefit restructuring' — a positive signal in LDA.")
            print("  This may reduce topic coherence for negated financial statements.")
        else:
            print("  No stop-words removed; all words including function words are in the vocabulary.")

        print(f"\n  Concrete example:")
        print(f"  Raw text:          'The company did not benefit from the restructuring'")
        print(f"  NLTK standard:     'company benefit restructuring'  (looks POSITIVE to LDA)")
        print(f"  Finance-adjusted:  'company not benefit restructuring' (negation preserved)")

        _cp2_done = True
        print("\nCP2 complete. Run the CP2 Feedback cell, then proceed to CP3.")


_cp2_btn.on_click(_run_cp2)
display(widgets.VBox([_cp2_norm_w, _cp2_sw_w, _cp2_num_w, _cp2_lower_w, _cp2_btn, _cp2_out]))


In [ ]:
#@title CP2 — Feedback

validate_checkpoint(["_cp2_done"], "CP2")

_fb = {
    "None": (
        "**No normalisation** preserves surface forms. This maximises transparency but inflates "
        "vocabulary with variants (e.g. *fund*, *funding*, *funded* treated as separate types). "
        "For a topic model, this increases sparsity without adding information."
    ),
    "Porter Stemmer": (
        "**Porter Stemming** aggressively truncates suffixes (*controversy* \u2192 *controversi*). "
        "It reduces vocabulary size efficiently but produces non-words that complicate human "
        "interpretation of topic word-lists \u2014 a significant cost when you need students to "
        "label topics meaningfully."
    ),
    "Lemmatisation (WordNetLemmatizer)": (
        "**Lemmatisation** reduces to dictionary base forms (*employees* \u2192 *employee*, "
        "*declining* \u2192 *decline*). It achieves vocabulary compression while preserving "
        "interpretable roots. For topic modelling of news text, this is generally the strongest choice."
    ),
}
_sw_fb = {
    "NLTK standard": (
        "The **standard NLTK list** removes function words efficiently. However, it also removes "
        "modals (*may*, *could*, *will*) and negation (*not*, *no*) \u2014 words that carry "
        "critical sentiment signal in financial coverage."
    ),
    "Finance-adjusted (retain modals & negation)": (
        "The **finance-adjusted list** retains modal verbs and negation terms. This is the better "
        "choice for sentiment analysis because it preserves uncertainty language (*may face "
        "insolvency*) and negation (*did not recover*) that dictionaries and classifiers depend on."
    ),
    "None": (
        "**No stop-word removal** leaves function words in the vocabulary. This adds noise to "
        "topic models and dilutes sentiment signal. It is rarely the right choice for analysis "
        "of news text."
    ),
}

norm_choice = session_data.get("cp2_normalisation", "")
sw_choice = session_data.get("cp2_stopwords", "")
norm_text = _fb.get(norm_choice, "")
sw_text = _sw_fb.get(sw_choice, "")

display(Markdown(
    f"### CP2 Feedback \u2014 Pre-processing choices and their consequences\n\n"
    f"{norm_text}\n\n"
    f"{sw_text}\n\n"
    "**Note on context:** These choices were originally designed for formal financial disclosures "
    "(10-K filings). News text has shorter documents, more colloquial language, and higher topical "
    "diversity. The finance-adjusted stop-word list retains its value here because modals and "
    "negation still carry critical sentiment signal in financial journalism."
))
print("Proceed to CP3.")


In [ ]:
#@title CP3 — Topic modelling: LDA by narrative era + STM-style temporal analysis

validate_checkpoint(["_cp2_done"], "CP2")

# BrewDog's four narrative eras (based on known arc)
ERAS = {
    "Growth (2010\u20132014)":          (2010, 2014),
    "Scaling (2015\u20132018)":         (2015, 2018),
    "Controversy (2019\u20132021)":     (2019, 2021),
    "Decline (2022\u20132023)":         (2022, 2023),
    "Administration (2024\u20132025)":  (2024, 2025),
}

# Extended stop words for LDA (prevents uninformative frequent tokens leaking
# through regardless of pre-processing choices made in CP2)
_LDA_EXTRA_STOPS = {
    "one", "two", "three", "four", "five", "also", "said", "say", "year",
    "new", "get", "got", "make", "made", "come", "came", "use", "used",
    "well", "still", "even", "just", "back", "way", "day", "time", "good",
    "last", "first", "tell", "told", "take", "took", "see", "saw", "know",
    "put", "set", "per", "cent", "per",
}

_cp3_ntopics_w = widgets.IntSlider(
    value=6, min=3, max=12, step=1,
    description="Topics (k):",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
    continuous_update=False,
)
_cp3_out = widgets.Output()
_cp3_run_btn = widgets.Button(
    description="Run LDA",
    button_style="primary",
    layout=widgets.Layout(width="150px"),
)
_cp3_label_boxes = []
_cp3_label_box_container = widgets.VBox([])
_cp3_label_btn = widgets.Button(
    description="Save topic labels",
    button_style="success",
    layout=widgets.Layout(width="180px"),
    disabled=True,
)
_cp3_era_btn = widgets.Button(
    description="Compare eras",
    button_style="info",
    layout=widgets.Layout(width="160px"),
    disabled=True,
)
_cp3_stm_btn = widgets.Button(
    description="STM-style analysis",
    button_style="warning",
    layout=widgets.Layout(width="190px"),
    disabled=True,
)


# ── Step 1: Run global LDA ─────────────────────────────────────────────────
def _run_cp3_lda(b):
    global _lda_model, _lda_vectorizer, _doc_topics, _cp3_run_done
    _cp3_out.clear_output(wait=True)
    with _cp3_out:
        k = _cp3_ntopics_w.value
        session_data["cp3_n_topics"] = k
        print(f"Fitting LDA with k={k} topics on {len(processed_docs)} documents...")
        _lda_vectorizer = CountVectorizer(max_features=3000, min_df=2, max_df=0.85,
            token_pattern=r'(?u)\b[a-z]{3,}\b', stop_words=list(_LDA_EXTRA_STOPS))
        dtm = _lda_vectorizer.fit_transform(processed_docs)
        _lda_model = LatentDirichletAllocation(
            n_components=k, random_state=42, learning_method="batch", max_iter=30,
        )
        _lda_model.fit(dtm)
        _doc_topics = _lda_model.transform(dtm)
        working_articles["dominant_topic"] = _doc_topics.argmax(axis=1)

        vocab = _lda_vectorizer.get_feature_names_out()
        print(f"\nTop 10 words per topic (global model — full corpus):\n")
        _top_sets = []
        for i, comp in enumerate(_lda_model.components_):
            top_words = [vocab[j] for j in comp.argsort()[-10:][::-1]]
            print(f"  Topic {i}: {', '.join(top_words)}")
            _top_sets.append(set(top_words))

        # --- Topic distinctiveness score (mean pairwise Jaccard on top-10 word sets) ---
        if k > 1:
            _overlaps = []
            for _ti in range(k):
                for _tj in range(_ti + 1, k):
                    _inter = len(_top_sets[_ti] & _top_sets[_tj])
                    _union = len(_top_sets[_ti] | _top_sets[_tj])
                    _overlaps.append(_inter / _union if _union > 0 else 0.0)
            _mean_ov = sum(_overlaps) / len(_overlaps) * 100
            print(f"\nTopic overlap (mean pairwise Jaccard on top-10 words): {_mean_ov:.1f}%")
            if _mean_ov > 30:
                print(f"  Overlap > 30%: topics share many top words — consider increasing k.")
            elif _mean_ov < 5:
                print(f"  Overlap < 5%: topics are very distinct — consider whether k is too large.")
            else:
                print(f"  Topics appear reasonably distinct (lower overlap = more distinct).")

        print("\nEnter a short label for each topic below, then click Save topic labels.")
        _cp3_run_done = True
        _cp3_label_btn.disabled = False

        boxes = []
        for i in range(k):
            comp = _lda_model.components_[i]
            top_w = [vocab[j] for j in comp.argsort()[-5:][::-1]]
            box = widgets.Text(
                value=f"Topic {i}",
                description=f"Topic {i} ({', '.join(top_w[:3])}):",
                style={"description_width": "280px"},
                layout=widgets.Layout(width="600px"),
            )
            boxes.append(box)
        _cp3_label_boxes[:] = boxes
        _cp3_label_box_container.children = (
            [widgets.HTML("<b>Enter topic labels (read the top words above first):</b>")] + boxes
        )


# ── Step 2: Save labels ────────────────────────────────────────────────────
def _save_labels(b):
    global _cp3_done
    _cp3_out.clear_output(wait=True)
    with _cp3_out:
        labels = [
            w.value.strip() or f"Topic {i}"
            for i, w in enumerate(_cp3_label_boxes)
        ]
        label_map = {i: lbl for i, lbl in enumerate(labels)}
        working_articles["topic_label"] = working_articles["dominant_topic"].map(label_map)
        session_data["cp3_topic_labels"] = "; ".join(f"{i}:{lbl}" for i, lbl in label_map.items())

        print("Topic labels saved:")
        for i, lbl in label_map.items():
            print(f"  Topic {i} → {lbl}")
        _cp3_done = True
        _cp3_era_btn.disabled = False
        _cp3_stm_btn.disabled = False
        print("\nNext steps:")
        print("  'Compare eras'      — see how topics differ across BrewDog's four narrative periods")
        print("  'STM-style analysis' — see how topic prevalence changes over time")


# ── Step 3: Era comparison ─────────────────────────────────────────────────
def _run_era_comparison(b):
    with _cp3_out:
        print("\n" + "─" * 60)
        print("Era comparison: LDA fitted separately per narrative period")
        print("─" * 60)

        era_k = max(3, min(_cp3_ntopics_w.value, 4))
        era_top_terms = {}

        for era_name, (yr_start, yr_end) in ERAS.items():
            era_df = working_articles[
                (working_articles["year"] >= yr_start) &
                (working_articles["year"] <= yr_end)
            ]
            era_texts = era_df["token_str"].tolist()

            if len(era_texts) < 5:
                era_top_terms[era_name] = []
                print(f"  {era_name}: insufficient articles ({len(era_texts)}) — skipped.")
                continue

            vect_era = CountVectorizer(max_features=1500, min_df=1, max_df=0.95,
                token_pattern=r'(?u)\b[a-z]{3,}\b', stop_words=list(_LDA_EXTRA_STOPS))
            dtm_era = vect_era.fit_transform(era_texts)
            lda_era = LatentDirichletAllocation(
                n_components=era_k, random_state=42, max_iter=25,
            )
            lda_era.fit(dtm_era)
            vocab_era = vect_era.get_feature_names_out()

            # Aggregate scores across all topics, deduplicate
            term_scores = {}
            for comp in lda_era.components_:
                for j in comp.argsort()[-15:][::-1]:
                    w = vocab_era[j]
                    term_scores[w] = term_scores.get(w, 0) + comp[j]
            top_terms = [w for w, _ in sorted(term_scores.items(), key=lambda x: -x[1])[:12]]
            era_top_terms[era_name] = top_terms

        # Print as a comparison table
        print()
        header = f"{'#':<4}" + "".join(f"{e:<26}" for e in ERAS)
        print(header)
        print("─" * len(header))
        n_rows = max((len(v) for v in era_top_terms.values()), default=0)
        for i in range(n_rows):
            row = f"{i+1:<4}"
            for era in ERAS:
                terms = era_top_terms.get(era, [])
                row += f"{terms[i] if i < len(terms) else '':<26}"
            print(row)

        # Heatmap: raw term frequency per era for the top terms found
        all_terms = []
        for terms in era_top_terms.values():
            for t in terms[:8]:
                if t not in all_terms:
                    all_terms.append(t)
        all_terms = all_terms[:24]

        heat_rows, era_labels = [], []
        for era_name, (yr_start, yr_end) in ERAS.items():
            era_df = working_articles[
                (working_articles["year"] >= yr_start) &
                (working_articles["year"] <= yr_end)
            ]
            era_text = " ".join(era_df["token_str"].tolist())
            tokens = era_text.split()
            total = max(len(tokens), 1)
            heat_rows.append([tokens.count(t) / total * 1000 for t in all_terms])
            era_labels.append(era_name)

        heat_df = pd.DataFrame(heat_rows, index=era_labels, columns=all_terms)

        fig, ax = plt.subplots(figsize=(14, 3.5))
        sns.heatmap(
            heat_df, ax=ax, cmap="YlOrRd",
            annot=True, fmt=".1f", linewidths=0.4,
            cbar_kws={"label": "Occurrences per 1,000 tokens"},
        )
        ax.set_title(
            "Key LDA terms by narrative era  (frequency per 1,000 tokens)\n"
            "Bright cells = term was disproportionately prominent in that era"
        )
        plt.xticks(rotation=45, ha="right", fontsize=8)
        plt.tight_layout()
        plt.show()

        print("\nLook for terms concentrated in a single era — those are era-defining topics.")
        print("Terms that appear across all eras are corpus-wide background vocabulary.")


# ── Step 4: STM-style temporal analysis ───────────────────────────────────
def _run_stm_style(b):
    with _cp3_out:
        print("\n" + "─" * 60)
        print("STM-style analysis: topic prevalence over time")
        print("─" * 60)

        label_map_rev = {f"Topic {i}": w.value.strip() or f"Topic {i}"
                         for i, w in enumerate(_cp3_label_boxes)}
        topic_cols = [f"Topic {i}" for i in range(_lda_model.n_components)]
        props_df = pd.DataFrame(_doc_topics, columns=topic_cols)
        props_df["year"] = working_articles["year"].values
        yr_props = props_df.groupby("year")[topic_cols].mean()
        yr_props.columns = [label_map_rev.get(c, c) for c in yr_props.columns]

        # OLS slope per topic (proportion ~ year)
        years_arr = yr_props.index.values.astype(float)
        years_c = years_arr - years_arr.mean()
        print("\nTopic prevalence trend (OLS: proportion ~ year):\n")
        slopes = {}
        for col in yr_props.columns:
            slope, _ = np.polyfit(years_c, yr_props[col].values, 1)
            direction = "↑ rising" if slope > 0.005 else ("↓ declining" if slope < -0.005 else "→ stable")
            slopes[col] = slope
            print(f"  {col:<32} slope = {slope:+.5f}   {direction}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        yr_props.plot(ax=axes[0], marker="o", colormap="tab10", linewidth=1.8)
        # Shade narrative eras
        yr_min, yr_max = years_arr.min(), years_arr.max()
        for shade_start, shade_end, color, label in [
            (2010, 2014.5, "#bbf7d0", "Growth"),
            (2014.5, 2018.5, "#fef08a", "Scaling"),
            (2018.5, 2021.5, "#fecaca", "Controversy"),
            (2021.5, 2025.5, "#fca5a5", "Crisis"),
        ]:
            if shade_end >= yr_min and shade_start <= yr_max:
                axes[0].axvspan(
                    max(shade_start, yr_min), min(shade_end, yr_max + 0.5),
                    alpha=0.18, color=color, zorder=0,
                )
        axes[0].set_title("Topic prevalence by year\n(mean document-topic proportion)")
        axes[0].set_xlabel("Year")
        axes[0].set_ylabel("Mean proportion")
        axes[0].legend(title="Topic", fontsize=7, bbox_to_anchor=(1.02, 1))

        slope_s = pd.Series(slopes).sort_values()
        bar_colors = ["#22c55e" if v > 0 else "#ef4444" for v in slope_s]
        slope_s.plot(kind="barh", ax=axes[1], color=bar_colors, edgecolor="white")
        axes[1].axvline(0, color="grey", linewidth=0.8)
        axes[1].set_title("Topic trend direction\n(OLS slope: positive = rising over time)")
        axes[1].set_xlabel("Slope coefficient (proportion ~ year)")
        plt.tight_layout()
        plt.show()

        display(Markdown(
            "---\n"
            "### What is Structural Topic Modelling (STM)?\n\n"
            "The analysis above is a manual approximation of what **Structural Topic Modelling (STM)** "
            "formalises statistically (Roberts, Stewart & Tingley, 2019).\n\n"
            "**What we just did — in four steps:**\n"
            "1. Fitted LDA on the full corpus to discover topics\n"
            "2. Extracted each document's topic proportion vector\n"
            "3. Grouped proportions by year to see how topic weight shifts over time\n"
            "4. Estimated a trend slope using OLS regression (proportion ~ year)\n\n"
            "**What STM does differently:** STM incorporates document-level metadata — such as `year`, "
            "`era`, or `firm type` — *inside* the model during estimation. Instead of discovering topics "
            "and *then* checking how year relates to them, STM uses year as a prevalence covariate when "
            "learning what the topics are. This means the topics themselves are already shaped by the "
            "temporal metadata, producing more temporally coherent themes than standard LDA.\n\n"
            "STM also supports **content covariates**: metadata can influence which *words* appear in a "
            "topic across groups. For example, an STM with `content ~ era` might reveal that the *Finance* "
            "topic uses different vocabulary in the Growth era (*crowdfunding*, *equity*, *punk*) versus "
            "the Crisis era (*administration*, *restructuring*, *going-concern*).\n\n"
            "> **Why we use the manual approximation here:** Python's scikit-learn LDA does not support "
            "STM natively. The canonical implementation is the R package `stm` (Roberts et al., 2019). "
            "The manual approach gives you the same interpretive output; the R version provides more "
            "rigorous statistical estimates and confidence intervals around covariate effects.\n\n"
            "**Key reference:** Roberts, M.E., Stewart, B.M., & Tingley, D. (2019). stm: An R package "
            "for structural topic models. *Journal of Statistical Software*, 91(2), 1–40."
        ))

        print("\nCP3 complete. Run the CP3 Feedback cell, then proceed to CP4.")


_cp3_run_btn.on_click(_run_cp3_lda)
_cp3_label_btn.on_click(_save_labels)
_cp3_era_btn.on_click(_run_era_comparison)
_cp3_stm_btn.on_click(_run_stm_style)

display(widgets.VBox([
    _cp3_ntopics_w,
    widgets.HBox([_cp3_run_btn, _cp3_label_btn, _cp3_era_btn, _cp3_stm_btn]),
    _cp3_label_box_container,
    _cp3_out,
]))


In [ ]:
#@title CP3 — Feedback

validate_checkpoint(["_cp3_done"], "CP3")

_k = session_data.get("cp3_n_topics", "?")
display(Markdown(
    f"### CP3 Feedback — Topic modelling: decisions, era comparison, and temporal structure\n\n"
    f"**Number of topics (k = {_k}).**  \n"
    "There is no statistically correct value of k for LDA; it is always a researcher decision. "
    "Too few topics conflates the *craft beer* narrative with *financial distress* simply because "
    "both appear in later-year articles. Too many topics fragments coherent themes into near-identical "
    "sub-topics and makes labelling unreliable. For a corpus of ~150 articles spanning 15 years, "
    "k between 5 and 8 typically balances coherence and coverage.\n\n"
    "**Why run LDA per era?**  \n"
    "A single LDA on the full corpus assumes that the same set of topics is present throughout — "
    "it can only learn that *administration* co-occurs with *losses* and *employees* if those words "
    "appear together in enough documents. When the corpus spans a dramatic narrative shift (craft beer "
    "success → reputational collapse → financial failure), a single global model blends vocabulary "
    "from incompatible periods. Fitting separate models per era allows each era's distinctive "
    "vocabulary to dominate without being diluted by unrelated periods.\n\n"
    "**Reading the era heatmap.**  \n"
    "Cells that are bright for one era and near-zero for others identify era-defining vocabulary. "
    "Terms that are uniformly present across all eras are corpus-wide background noise — words like "
    "*beer* or *company* tell you little about the narrative arc. The most analytically useful "
    "cells are those that show a clear temporal concentration.\n\n"
    "**STM and the value of metadata.**  \n"
    "The OLS slope table shows which topics are rising and which are declining over the 15-year "
    "period. A negative slope on a *Growth & Crowdfunding* topic and a positive slope on a "
    "*Controversy & Culture* topic would be consistent with what we know about BrewDog's arc. "
    "Structural Topic Modelling formalises this relationship by letting the year covariate shape "
    "topic discovery, rather than applying it retrospectively. The key interpretive advantage of "
    "STM in a case like BrewDog is that it can distinguish between topics that *always existed* "
    "but became more prominent over time, versus topics that genuinely *emerged* as new themes.\n\n"
    "**Ethical and methodological considerations:**\n\n"
    "1. **Topic labels are interpretations, not facts.** Two analysts given the same word list may "
    "assign different labels. Always report the top words alongside the label when presenting "
    "topic model results — the label is your interpretation of the words, not a ground truth.\n\n"
    "2. **Era boundaries are researcher choices.** The four eras used here (Growth / Scaling / "
    "Controversy / Crisis) were defined by knowledge of the BrewDog narrative. In a real analysis, "
    "you would not always have this prior knowledge. Consider how you would identify meaningful "
    "breakpoints from the data alone.\n\n"
    "3. **Training data coverage is uneven.** The 2021 open letter from 61 former employees "
    "generated a disproportionate volume of coverage in a short period. LDA will weight this "
    "burst of vocabulary heavily, potentially overstating the prominence of the workplace "
    "controversy topic relative to equally significant but less-covered issues.\n\n"
    "> **Question to consider:** If you applied the global LDA model to each era's documents "
    "separately (without re-fitting), would the dominant topics differ from those found by the "
    "era-specific models? What does this tell you about the difference between applying and "
    "estimating a topic model?"
))
print("Proceed to CP4.")

In [ ]:
#@title CP4 -- Sentiment analysis: dictionary and transformer methods

validate_checkpoint(["_cp3_done"], "CP3")

# BrewDog narrative themes for themed sentiment analysis
_THEMES = {
    "Leadership (James Watt)":    ["james watt", "watt", "chief executive", "co-founder", "ceo", "founder"],
    "Equity for Punks":           ["equity for punks", "crowdfunding", "crowd fund", "punk investor", "punk share"],
    "Financial distress":         ["administration", "going concern", "insolvency", "restructuring", "winding up", "losses"],
    "Workplace culture":          ["culture of fear", "former employee", "open letter", "allegations", "toxic", "mistreatment"],
}

# Newspaper groupings for source-bias analysis
_TABLOIDS     = {"The Sun", "Daily Mail", "Daily Mirror", "The Sun Online", "MailOnline"}
_BROADSHEETS  = {"The Times", "The Guardian", "The Daily Telegraph", "The Independent"}
_FINANCIAL    = {"FT.com", "Financial Times", "The Financial Times", "FT"}

def _source_type(paper):
    if paper in _TABLOIDS:     return "Tabloid"
    if paper in _BROADSHEETS:  return "Broadsheet"
    if paper in _FINANCIAL:    return "Financial press"
    return "Other"

# ── Widgets ─────────────────────────────────────────────────────────────────
_cp4_model_w = widgets.Dropdown(
    options=[
        "Dictionary only (VADER + LM, fast)",
        "FinBERT -- finance-tuned (recommended)  GPU: ~3 min  CPU: ~40 min",
        "DistilBERT-SST2 -- generic  GPU: ~2 min  CPU: ~12 min",
    ],
    value="FinBERT -- finance-tuned (recommended)  GPU: ~3 min  CPU: ~40 min",
    description="Method:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="640px"),
)
_cp4_timing_help = widgets.HTML(
    value=(
        "<span style='font-size:12px; color:#6b7280;'>"
        "Transformer inference runs on all 1,011 articles. "
        "For GPU speed: <b>Runtime \u2192 Change runtime type \u2192 T4 GPU</b> before clicking Run analysis. "
        "LM dictionary and VADER are always computed instantly on the full corpus."
        "</span>"
    )
)
_cp4_reflect_w = widgets.Dropdown(
    options=[
        "-- select after seeing the results --",
        "LM dictionary showed the earliest signal",
        "VADER showed the earliest signal",
        "FinBERT showed the earliest signal",
        "DistilBERT showed the earliest signal",
        "All methods agreed -- signal was consistent",
        "Methods diverged -- results were inconsistent",
        "None of the methods showed a clear early signal",
    ],
    description="Which method showed the earliest and clearest signal of distress?",
    style={"description_width": "420px"},
    layout=widgets.Layout(width="700px"),
    disabled=True,
)
_cp4_out = widgets.Output()
_cp4_run_btn      = widgets.Button(description="Run analysis",        button_style="primary",  layout=widgets.Layout(width="160px"))
_cp4_traj_btn     = widgets.Button(description="Sentiment over time",  button_style="info",     layout=widgets.Layout(width="190px"), disabled=True)
_cp4_paper_btn    = widgets.Button(description="By newspaper",         button_style="info",     layout=widgets.Layout(width="160px"), disabled=True)
_cp4_theme_btn    = widgets.Button(description="By theme",             button_style="warning",  layout=widgets.Layout(width="140px"), disabled=True)
_cp4_disagree_btn = widgets.Button(description="Show disagreements",   button_style="danger",   layout=widgets.Layout(width="180px"), disabled=True)
_cp4_lock_btn     = widgets.Button(description="Save responses",       button_style="success",  layout=widgets.Layout(width="160px"), disabled=True)

_cp4_transformer_results = [None]   # mutable container for cross-button access
_cp4_disagreements       = [None]   # top disagreement articles stored for CP5

# ── Step 1: Run analysis ─────────────────────────────────────────────────────
def _run_cp4_analysis(b):
    _cp4_out.clear_output(wait=True)
    with _cp4_out:
        choice = _cp4_model_w.value
        session_data["cp6_model"] = choice

        # Dictionary methods on all articles
        print("Computing LM and VADER sentiment on full corpus...")
        vader = SentimentIntensityAnalyzer()
        _lm_neg_url = globals().get("LM_NEG_URL", "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week07/data/lm_negative.txt")
        _lm_pos_url = globals().get("LM_POS_URL", "https://raw.githubusercontent.com/iamjamesbowden/AG952/main/materials/week07/data/lm_positive.txt")
        lm_neg = load_wordlist(_lm_neg_url)
        lm_pos = load_wordlist(_lm_pos_url)

        def _lm_score(tokens):
            p = sum(1 for t in tokens if t.lower() in lm_pos)
            n = sum(1 for t in tokens if t.lower() in lm_neg)
            return (p - n) / (p + n) if (p + n) > 0 else 0.0

        working_articles["vader_score"] = working_articles["text"].apply(
            lambda t: vader.polarity_scores(str(t)[:800])["compound"]
        )
        working_articles["lm_score"] = working_articles["tokens"].apply(_lm_score)
        working_articles["source_type"] = working_articles["newspaper"].apply(_source_type)

        # Capture early/late period means
        early = working_articles[working_articles["year"] <= 2014]["vader_score"].mean()
        late  = working_articles[working_articles["year"] >= 2019]["vader_score"].mean()
        session_data["cp4_sentiment_2010_2014"] = round(early, 4)
        session_data["cp4_sentiment_2019_2025"] = round(late, 4)

        print(f"  VADER mean 2010\u20132014:  {early:+.4f}")
        print(f"  VADER mean 2019\u20132025:  {late:+.4f}")
        print(f"  LM mean 2010\u20132014:     {working_articles[working_articles['year'] <= 2014]['lm_score'].mean():+.4f}")
        print(f"  LM mean 2019\u20132025:     {working_articles[working_articles['year'] >= 2019]['lm_score'].mean():+.4f}")

        # Transformer (if selected)
        if "Dictionary only" not in choice:
            if "FinBERT" in choice:
                model_id = "ProsusAI/finbert"
                label_map = {"positive": "positive", "negative": "negative", "neutral": "neutral"}
            else:
                model_id = "distilbert-base-uncased-finetuned-sst-2-english"
                label_map = {"POSITIVE": "positive", "NEGATIVE": "negative"}

            # Suppress the noisy LOAD REPORT that transformers prints for FinBERT
            import logging
            logging.getLogger("transformers").setLevel(logging.ERROR)

            print(f"\nLoading {model_id}...")
            from transformers import pipeline as hf_pipeline
            pipe = hf_pipeline("text-classification", model=model_id,
                                truncation=True, max_length=128, device=-1)

            trans_df = working_articles.copy()
            texts = (trans_df["headline"] + ". " + trans_df["text"].str[:250]).tolist()
            n_total = len(texts)
            print(f"Running inference on all {n_total} articles (progress below)...")

            # Chunked inference so progress is visible during the wait
            _chunk_size = 100
            preds = []
            for _i in range(0, n_total, _chunk_size):
                _chunk_preds = pipe(texts[_i:_i + _chunk_size], batch_size=16)
                preds.extend(_chunk_preds)
                print(f"  {min(_i + _chunk_size, n_total):>4}/{n_total} articles processed...")

            trans_df["transformer_label"] = [label_map.get(p["label"].lower(), "neutral") for p in preds]
            trans_df["transformer_score"] = [p["score"] for p in preds]

            # Normalise score: positive -> +score, negative -> -score, neutral -> 0
            trans_df["transformer_norm"] = trans_df.apply(
                lambda r: r["transformer_score"] if r["transformer_label"] == "positive"
                          else (-r["transformer_score"] if r["transformer_label"] == "negative" else 0.0),
                axis=1
            )
            _cp4_transformer_results[0] = trans_df

            # VADER-transformer agreement
            vader_label = trans_df["vader_score"].apply(
                lambda s: "positive" if s > 0.05 else ("negative" if s < -0.05 else "neutral")
            )
            if "DistilBERT" in choice:
                vader_cmp = vader_label.apply(lambda x: "positive" if x == "positive" else "negative")
            else:
                vader_cmp = vader_label
            agree_rate = (trans_df["transformer_label"] == vader_cmp).mean()
            if "FinBERT" in choice:
                session_data["cp6_finbert_accuracy"] = round(agree_rate, 4)
            else:
                session_data["cp6_distilbert_accuracy"] = round(agree_rate, 4)
            print(f"\n  VADER-transformer agreement rate: {agree_rate:.1%}")
            _cp4_disagree_btn.disabled = False
        else:
            _cp4_transformer_results[0] = None

        print("\nAnalysis complete. Use the buttons below to explore the results.")
        _cp4_traj_btn.disabled = False
        _cp4_paper_btn.disabled = False
        _cp4_theme_btn.disabled = False
        _cp4_reflect_w.disabled = False
        _cp4_lock_btn.disabled = False

# ── Step 2: Sentiment trajectories ─────────────────────────────────────────
def _run_cp4_trajectories(b):
    with _cp4_out:
        print("\n" + "-" * 60)
        print("Sentiment trajectories: all methods compared")
        print("-" * 60)

        yr_vader = working_articles.groupby("year")["vader_score"].mean()
        yr_lm    = working_articles.groupby("year")["lm_score"].mean()

        fig, ax = plt.subplots(figsize=(13, 5))
        ax.plot(yr_vader.index, yr_vader, marker="o", color="#4f46e5", linewidth=2,
                label="VADER (news-calibrated)", zorder=4)
        ax.plot(yr_lm.index, yr_lm, marker="s", color="#f59e0b", linewidth=2,
                linestyle="--", label="Loughran-McDonald (financial)", zorder=4)

        if _cp4_transformer_results[0] is not None:
            trans_df   = _cp4_transformer_results[0]
            model_name = session_data.get("cp6_model", "Transformer").split(" --")[0].strip()
            yr_trans   = trans_df.groupby("year")["transformer_norm"].mean()
            ax.plot(yr_trans.index, yr_trans, marker="^", color="#10b981", linewidth=2,
                    linestyle=":", label=f"{model_name} (n={len(trans_df)})", zorder=4)

        # Era shading
        for start, end, colour, label in [
            (2009.5, 2014.5, "#bbf7d0", "Growth"),
            (2014.5, 2018.5, "#fef3c7", "Scaling"),
            (2018.5, 2021.5, "#fed7aa", "Controversy"),
            (2021.5, 2023.5, "#fecaca", "Decline"),
            (2023.5, 2025.5, "#fca5a5", "Administration"),
        ]:
            ax.axvspan(start, end, alpha=0.25, color=colour, zorder=0, label=f"_{label}")
            ax.text((start + end) / 2, ax.get_ylim()[0] if ax.get_ylim()[0] != 0.0 else -0.3,
                    label, ha="center", va="bottom", fontsize=7, color="#6b7280")

        ax.axhline(0, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.set_title("Mean sentiment score by year -- all methods\n"
                     "(LM and VADER: full corpus; transformer: full corpus)")
        ax.set_xlabel("Year")
        ax.set_ylabel("Sentiment score (direction comparable, not scale)")
        ax.legend(fontsize=9, loc="lower left")
        plt.tight_layout()
        plt.show()

        # Identify turning point
        vader_vals = yr_vader.reset_index()
        neg_start = vader_vals[vader_vals["vader_score"] < 0]["year"].min()
        if not pd.isna(neg_start):
            print(f"\nFirst year VADER turns negative: {int(neg_start)}")
        lm_vals = yr_lm.reset_index()
        lm_neg_start = lm_vals[lm_vals["lm_score"] < 0]["year"].min()
        if not pd.isna(lm_neg_start):
            print(f"First year LM turns negative:    {int(lm_neg_start)}")
        print("These dates address RQ2: does one method signal distress earlier than another?")

# ── Step 3: Sentiment by newspaper ─────────────────────────────────────────
def _run_cp4_newspaper(b):
    with _cp4_out:
        print("\n" + "-" * 60)
        print("Sentiment by newspaper and source type")
        print("-" * 60)

        # Get top 10 newspapers with enough articles (at least 10)
        top_papers = working_articles["newspaper"].value_counts()
        top_papers = top_papers[top_papers >= 10].head(10)

        paper_means = (
            working_articles[working_articles["newspaper"].isin(top_papers.index)]
            .groupby("newspaper")["vader_score"]
            .agg(["mean", "sem", "count"])
            .reindex(top_papers.index)
        )

        source_colors = {
            "Tabloid": "#f97316",
            "Broadsheet": "#4f46e5",
            "Financial press": "#10b981",
            "Other": "#94a3b8",
        }
        bar_c = [source_colors[_source_type(p)] for p in paper_means.index]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: mean VADER by newspaper with error bars
        axes[0].barh(
            paper_means.index[::-1], paper_means["mean"][::-1],
            xerr=paper_means["sem"][::-1],
            color=bar_c[::-1], edgecolor="white", linewidth=0.5, capsize=3,
        )
        axes[0].axvline(0, color="grey", linewidth=0.8, linestyle="--")
        for src, colour in source_colors.items():
            axes[0].barh(0, 0, color=colour, label=src)  # legend proxies
        axes[0].legend(title="Source type", fontsize=8, loc="lower right")
        axes[0].set_title("Mean VADER sentiment by newspaper\n(bars = 95% confidence interval)")
        axes[0].set_xlabel("Mean VADER compound score")
        axes[0].tick_params(axis="y", labelsize=8)

        # Right: mean sentiment by source type over time
        wk = working_articles[working_articles["newspaper"].isin(top_papers.index)].copy()
        for src_type, colour in [("Tabloid", "#f97316"), ("Broadsheet", "#4f46e5"), ("Financial press", "#10b981")]:
            subset = wk[wk["source_type"] == src_type]
            if len(subset) > 0:
                yr_s = subset.groupby("year")["vader_score"].mean()
                axes[1].plot(yr_s.index, yr_s, marker="o", color=colour,
                             label=f"{src_type} (n={len(subset)})", linewidth=1.8)
        axes[1].axhline(0, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)
        axes[1].set_title("VADER sentiment by source type over time")
        axes[1].set_xlabel("Year")
        axes[1].set_ylabel("Mean VADER compound score")
        axes[1].legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        print("\nDo tabloids and the financial press tell the same story?")
        print("Look for divergence in the 2019-2021 period, when the culture controversy")
        print("(well-covered by tabloids) and the financial difficulties (well-covered by FT)")
        print("were overlapping but distinct narratives.")
        print("\nInterpretation caveat: mean scores by newspaper are not era-controlled.")
        print("Check whether newspapers with lower scores also published more heavily during")
        print("the Controversy and Decline eras before concluding that source type drove sentiment.")

# ── Step 4: Sentiment by theme ─────────────────────────────────────────────
def _run_cp4_theme(b):
    with _cp4_out:
        print("\n" + "-" * 60)
        print("Sentiment by topic theme")
        print("-" * 60)

        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        fig.suptitle("VADER sentiment trajectory by BrewDog narrative theme",
                     fontsize=12, fontweight="bold")

        theme_colors = ["#4f46e5", "#10b981", "#ef4444", "#f59e0b"]

        for ax, (theme_name, keywords), colour in zip(axes.flat, _THEMES.items(), theme_colors):
            mask_text = working_articles["text"].str.lower().apply(
                lambda t: any(k in t for k in keywords)
            )
            mask_head = working_articles["headline"].str.lower().apply(
                lambda h: any(k in h for k in keywords)
            )
            theme_df = working_articles[mask_text | mask_head].copy()

            if len(theme_df) < 5:
                ax.text(0.5, 0.5, f"Too few articles\n(n={len(theme_df)})",
                        ha="center", va="center", transform=ax.transAxes)
                ax.set_title(theme_name, fontweight="bold")
                continue

            yr_mean  = theme_df.groupby("year")["vader_score"].mean()
            yr_count = theme_df.groupby("year").size()

            ax2 = ax.twinx()
            ax2.bar(yr_count.index, yr_count.values, color=colour, alpha=0.15,
                    edgecolor="none", width=0.7)
            ax2.set_ylabel("Article count", fontsize=7, color="#94a3b8")
            ax2.tick_params(axis="y", labelsize=6, colors="#94a3b8")

            ax.plot(yr_mean.index, yr_mean.values, marker="o", color=colour,
                    linewidth=2, zorder=5)
            ax.axhline(0, color="grey", linestyle="--", linewidth=0.8, alpha=0.6, zorder=3)

            for start, end, shade in [(2009.5,2014.5,"#bbf7d0"),(2014.5,2018.5,"#fef3c7"),
                                       (2018.5,2021.5,"#fed7aa"),(2021.5,2023.5,"#fecaca"),(2023.5,2025.5,"#fca5a5")]:
                ax.axvspan(start, end, alpha=0.15, color=shade, zorder=0)

            ax.set_title(f"{theme_name}\n(n={len(theme_df)} articles, keywords: {', '.join(keywords[:3])}...)",
                         fontweight="bold", fontsize=9)
            ax.set_xlabel("Year", fontsize=8)
            ax.set_ylabel("Mean VADER score", fontsize=8)
            ax.tick_params(labelsize=7)

        plt.tight_layout()
        plt.show()

        print("\nEach panel shows the VADER sentiment for articles that mention a specific theme.")
        print("The 'Financial distress' theme should show the sharpest negative turn;")
        print("the 'Leadership' theme may reveal whether coverage of James Watt shifted")
        print("from celebratory to critical, and when.")

# ── Step 5: Show VADER-transformer disagreements ────────────────────────────
def _run_cp4_disagreements(b):
    with _cp4_out:
        print("\n" + "─" * 60)
        print("VADER-transformer disagreements: articles where methods diverge")
        print("─" * 60)

        if _cp4_transformer_results[0] is None:
            print("No transformer results. Run analysis with a transformer model first.")
            return

        sample_df = _cp4_transformer_results[0].copy()
        sample_df = sample_df[sample_df["vader_score"].abs() > 0.2].copy()
        if len(sample_df) == 0:
            print("No articles with |VADER compound| > 0.2 in this corpus.")
            return

        sample_df["vader_dir"] = sample_df["vader_score"].apply(
            lambda s: "positive" if s > 0 else "negative"
        )
        disagree = sample_df[
            (sample_df["transformer_label"] != "neutral") &
            (sample_df["vader_dir"] != sample_df["transformer_label"])
        ].copy()

        if len(disagree) == 0:
            print("No strong disagreements found.")
            print("(All articles with |VADER| > 0.2 received a consistent direction from the transformer.)")
            return

        disagree["gap"] = disagree["vader_score"].abs() + disagree["transformer_score"]

        vp_tn = disagree[
            (disagree["vader_dir"] == "positive") &
            (disagree["transformer_label"] == "negative")
        ].nlargest(5, "gap")

        vn_tp = disagree[
            (disagree["vader_dir"] == "negative") &
            (disagree["transformer_label"] == "positive")
        ].nlargest(5, "gap")

        def _print_table(df, title, explanation):
            if len(df) == 0:
                return
            print(f"\n{title} (n={len(df)}):")
            print(f"  {explanation}")
            print(f"  {'#':<4} {'Headline':<50} {'Yr':>4}  {'VADER':>7}  {'Transformer':>12}  {'Conf':>6}")
            print(f"  {'─' * 85}")
            for rank, (_, row) in enumerate(df.iterrows(), 1):
                hl = str(row.get("headline", ""))
                hl = hl[:47] + "..." if len(hl) > 47 else hl
                print(
                    f"  {rank:<4} {hl:<50} {int(row['year']):>4}  "
                    f"{row['vader_score']:>+7.3f}  {row['transformer_label']:>12}  "
                    f"{row['transformer_score']:>6.1%}"
                )

        _print_table(
            vp_tn,
            "VADER positive / transformer negative",
            "VADER saw positive language; FinBERT identified negative financial sentiment.",
        )
        _print_table(
            vn_tp,
            "VADER negative / transformer positive",
            "VADER saw negative language; FinBERT identified positive financial sentiment (e.g. risk mitigation).",
        )

        top_disagree = pd.concat([vp_tn, vn_tp]).head(5)
        _cp4_disagreements[0] = top_disagree
        print(f"\nTop {len(top_disagree)} disagreement articles stored for CP5 Button D.")
        print("In CP5, use Button D to run integrated gradients on them and understand why.")

# ── Step 6: Save responses ───────────────────────────────────────────────────
def _lock_cp4(b):
    global _cp4_done
    with _cp4_out:
        if _cp4_reflect_w.value.startswith("--"):
            print("Select a response to the reflection question before saving.")
            return
        session_data["cp4_dictionary"] = _cp4_reflect_w.value
        _cp4_done = True
        print(f"\nResponse saved: '{_cp4_reflect_w.value}'")
        print("CP4 complete. Run the CP4 Feedback cell, then proceed to Part 3.")

_cp4_run_btn.on_click(_run_cp4_analysis)
_cp4_traj_btn.on_click(_run_cp4_trajectories)
_cp4_paper_btn.on_click(_run_cp4_newspaper)
_cp4_theme_btn.on_click(_run_cp4_theme)
_cp4_disagree_btn.on_click(_run_cp4_disagreements)
_cp4_lock_btn.on_click(_lock_cp4)

display(widgets.VBox([
    _cp4_model_w,
    _cp4_timing_help,
    widgets.HBox([_cp4_run_btn, _cp4_traj_btn, _cp4_paper_btn, _cp4_theme_btn, _cp4_disagree_btn]),
    widgets.HTML("<br><b>After exploring the results, record your assessment:</b>"),
    _cp4_reflect_w,
    _cp4_lock_btn,
    _cp4_out,
]))


In [ ]:
#@title CP4 -- Feedback

validate_checkpoint(["_cp4_done"], "CP4")

_early = session_data.get("cp4_sentiment_2010_2014")
_late  = session_data.get("cp4_sentiment_2019_2025")
_early_str = f"{_early:+.4f}" if isinstance(_early, float) else str(_early)
_late_str  = f"{_late:+.4f}"  if isinstance(_late,  float) else str(_late)

display(Markdown(
    f"### CP4 Feedback -- Sentiment measurement: domain, timing and source\n\n"
    f"**Your numbers: early period {_early_str}, crisis period {_late_str}.**\n\n"
    "**The domain problem.**  \n"
    "The Loughran-McDonald dictionary was constructed from SEC 10-K filings. In that context, *liability*, "
    "*loss* and *default* are unambiguously negative. In newspaper coverage of a craft brewery, the same "
    "words appear in analysis, speculation and denial as well as in confirmed fact. LM will score "
    "*'analysts have raised the prospect of an insolvency filing'* as strongly negative, even though "
    "the sentence is reporting an analyst opinion rather than a corporate event. VADER, calibrated on "
    "news text and social media, handles this distinction somewhat better -- but still treats *administration* "
    "as a negative word regardless of whether it appears in *'entered administration'* or *'managed the "
    "risk of administration'*.\n\n"
    "**The timing problem.**  \n"
    "Look at the year when each method first turns negative. If VADER turns negative in 2021 and LM in "
    "2019, that is methodologically significant: it may mean LM picked up on financial risk language in "
    "business press coverage two years before the culture scandal drove the tabloid sentiment shift. Or "
    "it may mean LM over-counted neutral financial reporting as negative. Distinguishing between these "
    "interpretations requires reading the disagreement cases, which is exactly what CP5 will help you do.\n\n"
    "**The source problem.**  \n"
    "The corpus is dominated by The Sun (approximately 40 per cent of articles). The Sun's coverage of "
    "BrewDog is primarily consumer and entertainment-focused; it covers beer brands and celebrity "
    "associations rather than balance sheets. Its articles tend to be shorter, use more colloquial "
    "language, and are less likely to contain the financial terms that LM scores heavily. This means "
    "the overall corpus sentiment is skewed towards VADER-friendly language and away from "
    "LM-discernible financial risk signals. The newspaper breakdown in your CP4 visualisation "
    "should make this asymmetry visible.\n\n"
    "**What the theme analysis reveals.**  \n"
    "The four-theme breakdown is designed to separate BrewDog's overlapping negative narratives. "
    "The company faced simultaneous crises: a workplace culture scandal (peaking 2021), investor "
    "concerns about financial performance (emerging 2022-23), and eventual formal distress (2024-25). "
    "These are different stories with different vocabularies, and a single corpus-level sentiment score "
    "flattens them into a single number that is harder to interpret. Disaggregating by theme is a "
    "methodological choice with genuine analytical value.\n\n"
    "> **Question for your analyst note:** Given these measurement limitations, what additional "
    "validation would you require before presenting a sentiment-based analysis to an investment committee?"
))
print("Proceed to Part 3: Explainability.")


## Part 3 -- Explaining Transformer Predictions

### Why this matters

The trajectory charts in CP4 show that FinBERT and VADER sometimes disagree. When a transformer classifies an article as negative and a dictionary classifies it as neutral, one of them is wrong -- but without examining the evidence inside the model, it is difficult to say which.

**Integrated gradients** is a technique for attributing a model's output to its input tokens. It works by interpolating between a baseline input (all padding tokens -- a kind of 'neutral' input) and the actual sentence, measuring how the model's output changes at each step of the path. Tokens that, when changed, cause the biggest shift in the output score receive the highest attribution.

The result is a word-level importance score for each token. A word with a high positive attribution pushed the model towards its prediction; a word with a high negative attribution pushed it in the other direction. This is not the same as attention -- attention measures how much one token relies on another during processing, not how much it contributes to the final prediction. The distinction is important: a word can receive high attention without being a decisive driver of the output.

### Three sentences to watch

The five sentences in CP5 include two deliberate 'tricky' cases where dictionary methods are likely to misclassify but a well-calibrated transformer should not:

- Sentence 3: *'The company's auditors noted that the risk of administration had been materially reduced following last month's refinancing agreement.'* The words *risk* and *administration* are both LM-negative, but the sentence describes a positive outcome. Watch whether FinBERT and integrated gradients correctly weight *reduced* and *refinancing*.

- Sentence 4: *'The going-concern qualification included in last year's accounts was not repeated in the latest annual report.'* The LM dictionary will score *going-concern* and *qualification* as negative. The correct classification is neutral-to-positive (the qualification was removed). Watch how the model handles *not repeated*.


In [ ]:
#@title CP5 -- Explainability: integrated gradients on BrewDog sentences

validate_checkpoint(["_cp4_done"], "CP4")

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Five curated sentences spanning the BrewDog narrative ───────────────────
DEMO_SENTENCES = [
    {
        "label":  "Clear positive (growth phase)",
        "text":   "BrewDog has delivered its most profitable year on record, with revenues up 47 per cent driven by international expansion and record Equity for Punks investment.",
        "year":   2016,
    },
    {
        "label":  "Clear negative (crisis)",
        "text":   "BrewDog appointed administrators yesterday after failing to secure emergency refinancing, with creditors owed in excess of 300 million pounds.",
        "year":   2025,
    },
    {
        "label":  "Tricky -- mitigated risk (watch 'reduced' and 'refinancing')",
        "text":   "The company's auditors noted that the risk of administration had been materially reduced following last month's refinancing agreement.",
        "year":   2023,
    },
    {
        "label":  "Tricky -- going-concern qualification removed (watch 'not repeated')",
        "text":   "The going-concern qualification included in last year's accounts was not repeated in the latest annual report, following an improvement in liquidity.",
        "year":   2024,
    },
    {
        "label":  "Culture scandal (workplace, not financial)",
        "text":   "More than 60 former BrewDog employees published an open letter accusing the company of fostering a culture of fear and describing incidents of bullying and management misconduct.",
        "year":   2021,
    },
]

_CP5_MODEL = "ProsusAI/finbert"
_cp5_model_ref     = [None]
_cp5_tokenizer_ref = [None]
_cp5_explainer_ref = [None]

_cp5_out         = widgets.Output()
_cp5_load_btn    = widgets.Button(description="A: Load FinBERT",            button_style="primary", layout=widgets.Layout(width="200px"))
_cp5_explain_btn = widgets.Button(description="B: Explain with IG",         button_style="warning", layout=widgets.Layout(width="200px"), disabled=True)
_cp5_bar_btn     = widgets.Button(description="C: Attribution bar charts",  button_style="info",    layout=widgets.Layout(width="220px"), disabled=True)
_cp5_disagree_btn= widgets.Button(description="D: Explain disagreements",   button_style="danger",  layout=widgets.Layout(width="220px"), disabled=True)

_cp5_interp_w = widgets.Dropdown(
    options=[
        "-- select after viewing the attributions --",
        "Integrated gradients: clear and informative for financial text",
        "Integrated gradients: useful but hard to generalise from five sentences",
        "Attention weights: simpler to understand, close enough for practical use",
        "None of these: the model is too opaque for regulated financial analysis",
        "Need both methods together before drawing conclusions",
    ],
    description="Your assessment of integrated gradients as an analytical tool:",
    style={"description_width": "430px"},
    layout=widgets.Layout(width="700px"),
    disabled=True,
)
_cp5_lock_btn = widgets.Button(description="Save assessment", button_style="success",
                                layout=widgets.Layout(width="160px"), disabled=True)


# ── Button A: load model and show FinBERT predictions ─────────────────────
def _run_cp5_load(b):
    _cp5_out.clear_output(wait=True)
    with _cp5_out:
        print(f"Loading {_CP5_MODEL} (downloads ~420 MB on first run)...")
        tok   = AutoTokenizer.from_pretrained(_CP5_MODEL)
        model = AutoModelForSequenceClassification.from_pretrained(_CP5_MODEL)
        model.eval()
        _cp5_tokenizer_ref[0] = tok
        _cp5_model_ref[0]     = model

        _labels_map = {0: "positive", 1: "negative", 2: "neutral"}
        # FinBERT label ids vary -- use model.config.id2label
        id2label = model.config.id2label  # {0: 'positive', 1: 'negative', 2: 'neutral'} typically

        print(f"\n{'':>4} {'Sentence':<60} {'FinBERT':>10}  {'Confidence':>11}")
        print(f"  {'-'*85}")
        for i, s in enumerate(DEMO_SENTENCES):
            inputs = tok(s["text"], return_tensors="pt", truncation=True, max_length=128)
            with torch.no_grad():
                logits = model(**inputs).logits
            probs  = torch.softmax(logits, dim=-1)[0]
            pred_id = int(probs.argmax())
            pred_label = id2label.get(pred_id, str(pred_id))
            conf = float(probs.max())
            print(f"  {i+1}. {s['text'][:57]+'...':<60} {pred_label:>10}  {conf:>10.1%}")

        display(Markdown(
            "\n**Note on the tricky sentences:**  \n"
            "Sentences 3 and 4 contain LM-negative words (*risk*, *administration*, *going-concern*, "
            "*qualification*) but describe positive or neutral outcomes. Check whether FinBERT's "
            "prediction for those sentences matches the expected human label. A well-calibrated "
            "finance-tuned model should handle these correctly; a generic model would likely fail."
        ))
        _cp5_explain_btn.disabled = False

        # Enable Button D if CP4 disagreements are available
        _cp4_dis = globals().get("_cp4_disagreements", [None])[0]
        if _cp4_dis is not None and len(_cp4_dis) > 0:
            _cp5_disagree_btn.disabled = False
            print(f"\nButton D enabled: {len(_cp4_dis)} CP4 disagreement articles ready for IG analysis.")
        else:
            print("\nButton D: no CP4 disagreements found.")
            print("Run CP4 with a transformer model and click 'Show disagreements' to enable Button D.")

        print("\nModel loaded. Click 'B: Explain with IG' to run integrated gradients.")


# ── Button B: run integrated gradients ─────────────────────────────────────
def _run_cp5_explain(b):
    with _cp5_out:
        try:
            from transformers_interpret import SequenceClassificationExplainer
            tok   = _cp5_tokenizer_ref[0]
            model = _cp5_model_ref[0]

            print("\nRunning integrated gradients on all five sentences (30-60 seconds)...")
            explainer = SequenceClassificationExplainer(model, tok)
            _cp5_explainer_ref[0] = explainer

            for i, s in enumerate(DEMO_SENTENCES):
                print(f"\nSentence {i+1}: {s['label']}")
                word_attributions = explainer(s["text"])
                html_output = explainer.visualize()
                display(HTML(
                    f"<div style='font-size:13px;line-height:1.6;margin:6px 0 12px 0;'>"
                    f"<b>S{i+1} ({s['year']}):</b> "
                    + html_output.data +
                    f"</div>"
                ))

            _cp5_bar_btn.disabled = False
            _cp5_interp_w.disabled = False
            _cp5_lock_btn.disabled = False
            print("\nColour key: green = positive attribution (pushed towards current prediction);")
            print("red = negative attribution (pushed against current prediction).")
            print("The intensity reflects magnitude. Click 'C' for bar charts of the top words.")

        except Exception as e:
            print(f"Integrated gradients failed: {e}")
            print("This may occur on CPU with limited memory. Proceed to the bar chart view.")
            _cp5_bar_btn.disabled = False
            _cp5_interp_w.disabled = False
            _cp5_lock_btn.disabled = False


# ── Button C: bar charts for sentences 3 and 4 (the tricky cases) ──────────
def _run_cp5_bar(b):
    with _cp5_out:
        tok   = _cp5_tokenizer_ref[0]
        model = _cp5_model_ref[0]

        try:
            from transformers_interpret import SequenceClassificationExplainer
            explainer = _cp5_explainer_ref[0] or SequenceClassificationExplainer(model, tok)

            # Focus on sentences 3 and 4 (indices 2 and 3) -- the analytically interesting cases
            target_indices = [2, 3]
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            fig.suptitle("Integrated gradient attributions -- tricky sentences\n"
                         "(positive attribution = pushed towards current prediction)",
                         fontsize=11, fontweight="bold")

            for ax, idx in zip(axes, target_indices):
                s = DEMO_SENTENCES[idx]
                word_attributions = explainer(s["text"])
                # word_attributions: list of (token, score) tuples
                attrs = [(w, v) for w, v in word_attributions
                         if w not in ("[CLS]", "[SEP]", "[PAD]")]
                # Take top 14 by absolute value
                top = sorted(attrs, key=lambda x: abs(x[1]), reverse=True)[:14]
                top_sorted = sorted(top, key=lambda x: x[1])
                words, scores = zip(*top_sorted)
                colors = ["#22c55e" if s > 0 else "#ef4444" for s in scores]
                ax.barh(range(len(words)), scores, color=colors, edgecolor="white", linewidth=0.5)
                ax.set_yticks(range(len(words)))
                ax.set_yticklabels(words, fontsize=9)
                ax.axvline(0, color="grey", linewidth=0.8)
                ax.set_title(f"S{idx+1}: {s['label'][:45]}\n(year: {s['year']})",
                             fontsize=9, fontweight="bold")
                ax.set_xlabel("Attribution score")

            plt.tight_layout()
            plt.show()

            display(Markdown(
                "**Reading the charts:**  \n"
                "Green bars (positive attribution) drove the model towards its predicted class. "
                "Red bars pushed against the prediction. For sentence 3, you should see *reduced* "
                "and *refinancing* as green bars offsetting the negative pull of *administration*. "
                "For sentence 4, *not* and *repeated* should carry the positive signal that explains "
                "why the qualification was read as removed rather than present.\n\n"
                "If the model got these sentences wrong in Button A, the bar charts will show "
                "*why* -- the misleading LM-negative words will dominate the attribution, "
                "illustrating precisely the failure mode that contextual embeddings are supposed to fix."
            ))

        except Exception as e:
            print(f"Bar chart generation failed: {e}")
            print("Try viewing the HTML attributions from Button B instead.")


# ── Button D: explain top CP4 disagreement articles ────────────────────────
def _run_cp5_disagree(b):
    with _cp5_out:
        # Check model is loaded
        tok   = _cp5_tokenizer_ref[0]
        model = _cp5_model_ref[0]
        if tok is None or model is None:
            print("Model not loaded. Click 'A: Load FinBERT' first.")
            return

        # Check CP4 disagreements are available
        _cp4_dis = globals().get("_cp4_disagreements", [None])[0]
        if _cp4_dis is None or len(_cp4_dis) == 0:
            print("No CP4 disagreements available.")
            print("Run CP4 with a transformer model and click 'Show disagreements', then return here.")
            return

        try:
            from transformers_interpret import SequenceClassificationExplainer
        except ImportError:
            print("transformers_interpret not available. Ensure it was installed in Step 0.")
            return

        explainer = _cp5_explainer_ref[0] or SequenceClassificationExplainer(model, tok)
        _cp5_explainer_ref[0] = explainer

        n_articles = min(3, len(_cp4_dis))
        print("\n" + "─" * 60)
        print(f"Button D: integrated gradients on top {n_articles} CP4 disagreement articles")
        print("─" * 60)
        print("For each article, ask: do the highlighted tokens explain why FinBERT disagreed")
        print("with VADER? Does the model focus on the right words?\n")

        for rank, (_, row) in enumerate(_cp4_dis.head(n_articles).iterrows(), 1):
            hl   = str(row.get("headline", ""))
            text = str(row.get("text", ""))
            # Use same input format as CP4 analysis
            input_text = (hl + ". " + text[:250]) if hl else text[:280]
            vader_s    = row.get("vader_score", 0)
            trans_lbl  = row.get("transformer_label", "?")
            trans_conf = row.get("transformer_score", 0)

            print(f"Article {rank}: {hl[:72]}")
            print(f"  Year: {int(row['year'])}  |  VADER: {vader_s:+.3f}  |  FinBERT: {trans_lbl} ({trans_conf:.1%})")

            try:
                word_attributions = explainer(input_text)
                html_output = explainer.visualize()
                display(HTML(
                    f"<div style='font-size:13px;line-height:1.6;margin:6px 0 16px 0;'>"
                    + html_output.data +
                    f"</div>"
                ))
            except Exception as e:
                print(f"  IG failed for article {rank}: {e}")

        print("\nColour key: green = attribution towards predicted class; red = attribution away.")
        print("Identify the tokens driving the disagreement — are they the words you would")
        print("expect a domain-aware model to prioritise over a general lexical approach?")


# ── Lock in response ─────────────────────────────────────────────────────────
def _lock_cp5(b):
    global _cp5_done
    with _cp5_out:
        if _cp5_interp_w.value.startswith("--"):
            print("Select your assessment before saving.")
            return
        session_data["cp7_interpretability_choice"] = _cp5_interp_w.value
        _cp5_done = True
        print(f"\nSaved: '{_cp5_interp_w.value}'")
        print("CP5 complete. Proceed to the CP5 Feedback cell, then Part 4.")


_cp5_load_btn.on_click(_run_cp5_load)
_cp5_explain_btn.on_click(_run_cp5_explain)
_cp5_bar_btn.on_click(_run_cp5_bar)
_cp5_disagree_btn.on_click(_run_cp5_disagree)
_cp5_lock_btn.on_click(_lock_cp5)

display(widgets.VBox([
    widgets.HBox([_cp5_load_btn, _cp5_explain_btn, _cp5_bar_btn, _cp5_disagree_btn]),
    _cp5_out,
    widgets.HTML("<br><b>After reviewing the attributions, record your assessment:</b>"),
    _cp5_interp_w,
    _cp5_lock_btn,
]))


In [ ]:
#@title CP5 -- Feedback

validate_checkpoint(["_cp5_done"], "CP5")

display(Markdown(
    "### CP5 Feedback -- What integrated gradients can and cannot tell you\n\n"
    "**What you just saw.**  \n"
    "For the clear cases (sentences 1 and 2), the attribution patterns are reassuringly interpretable: "
    "words like *profitable*, *revenues* and *expansion* drive positive predictions; *administrators*, "
    "*failing* and *creditors* drive negative ones. This is consistent with what a domain expert would "
    "expect, which is a form of face validity.\n\n"
    "**The tricky cases matter more.**  \n"
    "Sentences 3 and 4 are the test. If FinBERT correctly classifies the mitigated-risk sentence as "
    "neutral or positive, and the attribution chart shows *reduced* and *refinancing* as the key "
    "positive drivers, then the model has genuinely learned to handle context -- not just to recognise "
    "negative words. That is a meaningful result. If it still scores negative, the attributions will "
    "likely show *administration* dominating, demonstrating that the model has not learned to "
    "distinguish occurrence from mitigation.\n\n"
    "**The regulatory picture.**  \n"
    "The EU AI Act (2024) classifies systems used in financial analysis as high-risk and requires "
    "explainability of model outputs. Integrated gradients satisfies this requirement more rigorously "
    "than attention weights, because it measures causal contribution to the output rather than "
    "contextual relevance. However, it still operates at the level of individual sentences rather "
    "than full articles, and attribution scores from five sentences cannot be generalised to "
    "the full corpus without substantial additional validation.\n\n"
    "**The difference between attention and gradients.**  \n"
    "Attention shows which tokens a given token looked at when building its contextual representation. "
    "Integrated gradients show which tokens caused the final classification. A token can receive "
    "high attention (the model looked at it a lot) without having high attribution (it did not "
    "change the output much). Jain and Wallace (2019) showed empirically that high attention "
    "does not reliably identify the tokens that most influence predictions. This is why integrated "
    "gradients, which respect the input-output relationship directly, are considered a more "
    "principled explainability method.\n\n"
    "> **For your analyst note:** Does the evidence from CP5 change your view of whether "
    "transformer-based sentiment is reliable enough for financial analysis? Be specific about "
    "what you observed."
))
print("Proceed to Part 4: submission.")


## Part 4 -- Analyst's Note and Submission


## CP6 -- Analyst's Note


In [ ]:
#@title CP6 -- Analyst's note (150 words) and submission

validate_checkpoint(["_cp5_done"], "CP5")

display(Markdown(
    "### CP6 -- Analyst's Note\n\n"
    "You have worked through all three research questions:\n\n"
    "- **RQ1:** You examined how coverage volume and topic composition shifted across "
    "BrewDog's growth, controversy and crisis phases, and whether the LDA topic model "
    "recovered those shifts.\n"
    "- **RQ2:** You compared dictionary-based sentiment (LM, VADER) against a transformer "
    "(FinBERT or DistilBERT), observed where they agreed and diverged, and explored "
    "patterns by newspaper and by theme.\n"
    "- **RQ3:** You used integrated gradients to examine what drove FinBERT's predictions "
    "on five representative sentences, including two that challenge dictionary-based approaches.\n\n"
    "Write a **maximum 150-word analyst's note** addressed to your investment committee. "
    "The note should cover:\n\n"
    "1. Whether the text analysis methods, taken together, provide evidence of early warning "
    "signals in the corpus\n"
    "2. The most significant limitation that would affect the reliability of those signals\n"
    "3. What additional data or analysis you would require before including this in a "
    "formal investment memorandum\n\n"
    "Your note will be displayed to the class during the debrief."
))

_cp6_note_w = widgets.Textarea(
    placeholder="Write your analyst note here (max 150 words)...",
    layout=widgets.Layout(width="700px", height="160px"),
)
_cp6_wc_label = widgets.Label("Word count: 0 / 150")
_cp6_out = widgets.Output()
_cp6_btn = widgets.Button(
    description="Submit note",
    button_style="success",
    layout=widgets.Layout(width="180px"),
)

def _update_wc(change):
    wc = len(change["new"].split())
    _cp6_wc_label.value = f"Word count: {wc} / 150"
    _cp6_wc_label.style.text_color = "#ef4444" if wc > 150 else "#22c55e"

_cp6_note_w.observe(_update_wc, names="value")

def _submit(b):
    _cp6_out.clear_output(wait=True)
    with _cp6_out:
        note = _cp6_note_w.value.strip()
        wc = len(note.split())
        if wc < 20:
            print("Please write at least 20 words before submitting.")
            return
        if wc > 165:
            print(f"Your note is {wc} words. Please reduce to 150 words.")
            return

        session_data["cp8_analyst_note"] = note
        session_data["timestamp"] = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())

        payload = {k: str(v) if v is not None else "" for k, v in session_data.items()}

        if not APPS_SCRIPT_URL:
            print("APPS_SCRIPT_URL not set -- submission skipped (instructor has not configured endpoint).")
            print(f"\nYour session data:\n{json.dumps(payload, indent=2)}")
            return

        try:
            resp = requests.post(APPS_SCRIPT_URL, json=payload, timeout=20,
                                 headers={"Content-Type": "application/json"})
            result = resp.json()
            if result.get("status") == "success":
                print(f"Submitted -- team: {session_data['team_name']}")
                print(f"\nYour analyst note ({wc} words):")
                print(f'\n"{note}"')
                print("\nWait for the instructor to display the class debrief.")
            else:
                print(f"Submission error: {result.get('message', 'unknown')}")
                print("Note saved locally. Contact your instructor if the problem persists.")
        except Exception as ex:
            print(f"Submission failed: {ex}")
            print("Note saved locally. Contact your instructor.")

_cp6_btn.on_click(_submit)
display(widgets.VBox([_cp6_note_w, _cp6_wc_label, _cp6_btn, _cp6_out]))
